## Check what all is installed first

In [ ]:
import numpy, pandas, sklearn, matplotlib, seaborn
print(numpy.__version__, pandas.__version__, sklearn.__version__)


## Install the other required libraries

In [ ]:
# Install packages not pre-installed on Kaggle (most are already available)
!pip install -q nltk spacy gensim beautifulsoup4 rouge-score kaggle
!pip install -q transformers datasets accelerate evaluate rouge-score sentencepiece
print("Installed")

## Authentication with HuggingFace CLI

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

user_secrets = UserSecretsClient()

HUGGING_FACE_ACCESS_TOKEN = user_secrets.get_secret(
    "HUGGING_FACE_ACCESS_TOKEN"
)

print("Token loaded:", bool(HUGGING_FACE_ACCESS_TOKEN))
print("Token length:", len(HUGGING_FACE_ACCESS_TOKEN))

login(token=HUGGING_FACE_ACCESS_TOKEN)

print(whoami())

## Project Configurations

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from collections import Counter

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
        Trainer,
    EarlyStoppingCallback,
)

from datasets import load_dataset

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from rouge_score import rouge_scorer


# ============================================================
# RAW DATASET PATHS
# ============================================================

ENRON_DIR = Path(
    "/kaggle/input/datasets/wcukierski/enron-email-dataset"
)

SPAMASSASSIN_DIR = Path(
    "/kaggle/input/datasets/prishabhkumar/spamassassin"
)

EMAILSUM_DIR = Path(
    "/kaggle/input/datasets/prishabhkumar/emailsum"
)

BC3_DIR = Path(
    "/kaggle/input/datasets/prishabhkumar/bc3-corpus"
)

ENRON_CSV = ENRON_DIR / "emails.csv"


# ============================================================
# PERSISTENT PROCESSED DATASET
# ============================================================

PROCESSED_DATASET_DIR = Path(
    "/kaggle/input/datasets/prishabhkumar/email-intelligence-processed-data"
)

PROCESSED_DIR = PROCESSED_DATASET_DIR

CLEAN_DIR = PROCESSED_DATASET_DIR / "Cleaned data"
TRAIN_DIR = PROCESSED_DATASET_DIR / "train"
VAL_DIR = PROCESSED_DATASET_DIR / "val"
TEST_DIR = PROCESSED_DATASET_DIR / "test"


# ============================================================
# TEMPORARY KAGGLE WORKING DIRECTORY
# ============================================================

WORK_DIR = Path("/kaggle/working")

MODEL_DIR = WORK_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

WORK_PROCESSED_DIR = WORK_DIR / "data" / "processed"


# ============================================================
# EXPECTED PROCESSED FILES
# ============================================================

ENRON_CLEANED_CSV = CLEAN_DIR / "enron_cleaned.csv"
SPAMASSASSIN_CLEANED_CSV = CLEAN_DIR / "spamassassin_cleaned.csv"
BC3_JSON = CLEAN_DIR / "bc3_threads.json"

ENRON_TRAIN_CSV = TRAIN_DIR / "enron_train.csv"
ENRON_VAL_CSV = VAL_DIR / "enron_val.csv"
ENRON_TEST_CSV = TEST_DIR / "enron_test.csv"

SPAMASSASSIN_TRAIN_CSV = TRAIN_DIR / "spamassassin_train.csv"
SPAMASSASSIN_VAL_CSV = VAL_DIR / "spamassassin_val.csv"
SPAMASSASSIN_TEST_CSV = TEST_DIR / "spamassassin_test.csv"


# ============================================================
# INTENT LABEL MAPPING
# ============================================================

INTENT_LABELS = {
    0: "REQUEST",
    1: "FOLLOW_UP",
    2: "INFORMATION",
    3: "ACKNOWLEDGEMENT",
    4: "COMPLAINT",
    5: "INVITATION",
}

NUM_INTENT_LABELS = len(INTENT_LABELS)


# ============================================================
# TRAINING HYPERPARAMETERS
# ============================================================

INTENT_MODEL_NAME = "distilbert-base-uncased"
SUMMARIZATION_MODEL_NAME = "t5-small"

MAX_SEQ_LENGTH = 512
MAX_SUMMARY_LENGTH = 128


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")


# ============================================================
# VERIFY RAW DATASETS
# ============================================================

print("\n=== RAW DATASET PATHS ===")

print(f"Enron:        {ENRON_DIR}")
print(f"SpamAssassin: {SPAMASSASSIN_DIR}")
print(f"EMAILSUM:     {EMAILSUM_DIR}")
print(f"BC3:          {BC3_DIR}")

print("\n=== RAW DATASET EXISTENCE ===")

print(f"Enron exists:        {ENRON_DIR.exists()}")
print(f"SpamAssassin exists: {SPAMASSASSIN_DIR.exists()}")
print(f"EMAILSUM exists:     {EMAILSUM_DIR.exists()}")
print(f"BC3 exists:          {BC3_DIR.exists()}")

print("\n=== ENRON CSV ===")

print(f"Path:   {ENRON_CSV}")
print(f"Exists: {ENRON_CSV.exists()}")


# ============================================================
# VERIFY PERSISTENT PROCESSED DATASET
# ============================================================

print("\n=== PERSISTENT PROCESSED DATASET ===")

print(f"Dataset: {PROCESSED_DATASET_DIR}")
print(f"Exists:  {PROCESSED_DATASET_DIR.exists()}")


# ============================================================
# VERIFY PROCESSED DIRECTORIES
# ============================================================

print("\n=== PROCESSED DIRECTORIES ===")

print(f"CLEAN_DIR: {CLEAN_DIR}")
print(f"TRAIN_DIR: {TRAIN_DIR}")
print(f"VAL_DIR:   {VAL_DIR}")
print(f"TEST_DIR:  {TEST_DIR}")

print("\n=== PROCESSED DIRECTORY EXISTENCE ===")

print(f"CLEAN_DIR exists: {CLEAN_DIR.exists()}")
print(f"TRAIN_DIR exists: {TRAIN_DIR.exists()}")
print(f"VAL_DIR exists:   {VAL_DIR.exists()}")
print(f"TEST_DIR exists:  {TEST_DIR.exists()}")


# ============================================================
# VERIFY PROCESSED FILES
# ============================================================

print("\n=== PROCESSED FILES ===")

processed_files = {
    "Enron cleaned": ENRON_CLEANED_CSV,
    "SpamAssassin cleaned": SPAMASSASSIN_CLEANED_CSV,
    "BC3": BC3_JSON,
    "Enron train": ENRON_TRAIN_CSV,
    "Enron val": ENRON_VAL_CSV,
    "Enron test": ENRON_TEST_CSV,
    "SpamAssassin train": SPAMASSASSIN_TRAIN_CSV,
    "SpamAssassin val": SPAMASSASSIN_VAL_CSV,
    "SpamAssassin test": SPAMASSASSIN_TEST_CSV,
}

for name, path in processed_files.items():
    print(f"{name:25} -> {path.exists()}")


# ============================================================
# KAGGLE INPUT ROOT
# ============================================================

print("\n=== KAGGLE INPUT ===")
!ls /kaggle/input/

## Automatic Intent Candidate Generation
This will later be used to generate the intent_train, intent_test and intent_val datasets

In [ ]:
import re
import time
import pandas as pd
from pathlib import Path

INTENT_RULES = {
    0: {
        "name": "REQUEST",
        "patterns": [
            r"\bplease send\b",
            r"\bplease provide\b",
            r"\bcould you\b",
            r"\bcan you\b",
            r"\bwould you\b",
            r"\bwill you\b",
            r"\bplease let me know\b",
            r"\bplease confirm\b",
            r"\bplease review\b",
            r"\bplease forward\b",
            r"\bplease advise\b",
            r"\bneed you to\b",
            r"\bi need\b",
            r"\bwe need\b",
            r"\brequesting\b",
        ],
    },
    1: {
        "name": "FOLLOW_UP",
        "patterns": [
            r"\bfollowing up\b",
            r"\bfollow up\b",
            r"\bfollow-up\b",
            r"\bjust checking\b",
            r"\bchecking in\b",
            r"\bany update\b",
            r"\bany updates\b",
            r"\bstatus update\b",
            r"\bchecking on\b",
            r"\bcircling back\b",
            r"\bget back to me\b",
            r"\bget back\b",
            r"\breminder\b",
            r"\bremind you\b",
            r"\bprevious email\b",
            r"\bprevious message\b",
        ],
    },
    2: {
        "name": "INFORMATION",
        "patterns": [
            r"\bfor your information\b",
            r"\bfor your reference\b",
            r"\bfyi\b",
            r"\bplease note\b",
            r"\bi wanted to inform\b",
            r"\bi am writing to inform\b",
            r"\bthis is to inform\b",
            r"\bjust to let you know\b",
            r"\blet you know\b",
            r"\bhere is\b",
            r"\bhere are\b",
            r"\battached is\b",
            r"\battached are\b",
            r"\bregarding\b",
            r"\binformation about\b",
        ],
    },
    3: {
        "name": "ACKNOWLEDGEMENT",
        "patterns": [
            r"\bthanks\b",
            r"\bthank you\b",
            r"\bthanks for\b",
            r"\bthank you for\b",
            r"\breceived\b",
            r"\bwell received\b",
            r"\bgot it\b",
            r"\bunderstood\b",
            r"\bnoted\b",
            r"\bconfirmed\b",
            r"\bi confirm\b",
            r"\bwe confirm\b",
            r"\bagree\b",
            r"\bsounds good\b",
            r"\blooks good\b",
            r"\bwill do\b",
        ],
    },
    4: {
        "name": "COMPLAINT",
        "patterns": [
            r"\bcomplaint\b",
            r"\bcomplain\b",
            r"\bdisappointed\b",
            r"\bunacceptable\b",
            r"\bnot acceptable\b",
            r"\bpoor service\b",
            r"\bterrible service\b",
            r"\bfrustrated\b",
            r"\bfrustrating\b",
            r"\bproblem with\b",
            r"\bissue with\b",
            r"\bconcern about\b",
            r"\bconcerned about\b",
            r"\bnot happy\b",
            r"\bvery unhappy\b",
            r"\bdissatisfied\b",
            r"\bdissatisfaction\b",
        ],
    },
    5: {
        "name": "INVITATION",
        "patterns": [
            r"\byou are invited\b",
            r"\byou're invited\b",
            r"\binvitation\b",
            r"\binvite you\b",
            r"\binviting you\b",
            r"\bplease join us\b",
            r"\bjoin us for\b",
            r"\bjoin me for\b",
            r"\bwould like you to join\b",
            r"\bwelcome to\b",
            r"\bcome to\b",
            r"\battend the meeting\b",
            r"\battend our meeting\b",
            r"\bjoin the meeting\b",
        ],
    },
}

COMPILED_INTENT_RULES = {}

for label_id, rule in INTENT_RULES.items():
    COMPILED_INTENT_RULES[label_id] = {
        "name": rule["name"],
        "patterns": [
            re.compile(pattern, re.IGNORECASE)
            for pattern in rule["patterns"]
        ],
    }


def score_intent(text):
    text = str(text)

    scores = {}

    for label_id, rule in COMPILED_INTENT_RULES.items():
        score = 0

        for pattern in rule["patterns"]:
            score += sum(
                1 for _ in pattern.finditer(text)
            )

        scores[label_id] = score

    best_label = max(
        scores,
        key=scores.get
    )

    best_score = scores[best_label]

    sorted_scores = sorted(
        scores.values(),
        reverse=True
    )

    second_score = sorted_scores[1]

    if best_score == 0:
        return None, 0, scores

    confidence = best_score - second_score

    if confidence >= 1:
        return best_label, best_score, scores

    return None, best_score, scores


ENRON_INTENT_SOURCE = (
    Path("/kaggle/input/datasets/prishabhkumar/")
    / "email-intelligence-processed-data"
    / "Cleaned data"
    / "enron_cleaned.csv"
)

print("Loading:")
print(ENRON_INTENT_SOURCE)
print("Exists:", ENRON_INTENT_SOURCE.exists())

if not ENRON_INTENT_SOURCE.exists():
    raise FileNotFoundError(
        f"Enron dataset not found:\n{ENRON_INTENT_SOURCE}"
    )

enron_intent_df = pd.read_csv(
    ENRON_INTENT_SOURCE
)

total_emails = len(enron_intent_df)

print(f"\nTotal Enron emails: {total_emails:,}")
print("\nStarting intent candidate generation...\n")

results = []

start_time = time.time()
last_update = start_time
update_interval = 5

for index, (_, row) in enumerate(
    enron_intent_df.iterrows(),
    start=1
):

    subject = str(
        row.get("subject", "")
    )

    body = str(
        row.get("cleaned_body", "")
    )

    text = f"{subject} {body}"

    label, score, scores = score_intent(text)

    if label is not None:
        results.append({
            "file": row["file"],
            "sender": row["sender"],
            "date": row["date"],
            "subject": row["subject"],
            "cleaned_body": row["cleaned_body"],
            "intent_label": label,
            "intent_name": INTENT_RULES[label]["name"],
            "confidence_score": score,
        })

    current_time = time.time()

    if (
        current_time - last_update >= update_interval
        or index == total_emails
    ):
        elapsed = current_time - start_time

        rate = index / elapsed if elapsed > 0 else 0

        remaining = total_emails - index

        estimated_remaining = (
            remaining / rate
            if rate > 0
            else 0
        )

        progress = (
            index / total_emails
        ) * 100

        elapsed_minutes = elapsed / 60
        remaining_minutes = estimated_remaining / 60

        print(
            f"Progress: {index:,}/{total_emails:,} "
            f"({progress:.2f}%) | "
            f"Speed: {rate:.2f} emails/sec | "
            f"Elapsed: {elapsed_minutes:.2f} min | "
            f"Remaining: {remaining_minutes:.2f} min"
        )

        last_update = current_time


intent_candidates = pd.DataFrame(
    results
)

total_elapsed = time.time() - start_time

print("\n========== INTENT CANDIDATE RESULTS ==========")

print(
    f"High-confidence candidates: "
    f"{len(intent_candidates):,}"
)

print(
    f"Total processing time: "
    f"{total_elapsed / 60:.2f} minutes"
)

if total_elapsed > 0:
    print(
        f"Average processing speed: "
        f"{total_emails / total_elapsed:.2f} emails/sec"
    )

if len(intent_candidates) > 0:
    print("\nCandidate distribution:")

    print(
        intent_candidates[
            "intent_name"
        ]
        .value_counts()
        .sort_index()
    )

else:
    print(
        "\nNo high-confidence intent candidates "
        "were generated."
    )

## Saving the high-confidence candidates in a separate file

In [ ]:
INTENT_CANDIDATES_OUT = Path("/kaggle/working/data/processed/Cleaned data/intent_candidates.csv")

INTENT_CANDIDATES_OUT.parent.mkdir(parents=True, exist_ok=True)

intent_candidates.to_csv(
    INTENT_CANDIDATES_OUT,
    index=False
)

print(f"Saved: {INTENT_CANDIDATES_OUT}")
print(f"Rows: {len(intent_candidates):,}")

## Validation cell to inspect label distribution

In [ ]:
print("=== Intent Candidate Verification ===")

print(f"Total candidates: {len(intent_candidates):,}")

print("\n=== Label Counts ===")
print(
    intent_candidates["intent_name"]
    .value_counts()
)

print("\n=== Label Percentages ===")
print(
    (intent_candidates["intent_name"]
     .value_counts(normalize=True) * 100)
    .round(2)
)

print("\n=== Score Distribution ===")
print(
    intent_candidates["confidence_score"]
    .value_counts()
    .sort_index()
)

print("\n=== Sample Emails Per Intent ===")

for intent in INTENT_LABELS.values():
    subset = intent_candidates[
        intent_candidates["intent_name"] == intent
    ]

    print(f"\n{'=' * 60}")
    print(f"INTENT: {intent}")
    print(f"Count: {len(subset):,}")

    if len(subset) > 0:
        sample = subset.sample(
            min(3, len(subset)),
            random_state=42
        )

        for _, row in sample.iterrows():
            print("\nSubject:", row["subject"])
            print("Body:", str(row["cleaned_body"])[:500])
            print("Score:", row["confidence_score"])

## Stricter intent label thresohld to create the final intent_labeled.csv

## Verification of EmailSum data

In [ ]:
# Create final automatically labeled intent dataset

MIN_CONFIDENCE = 2

intent_labeled_df = intent_candidates[
    intent_candidates["confidence_score"] >= MIN_CONFIDENCE
].copy()

intent_labeled_df = intent_labeled_df.drop_duplicates(
    subset=["file"]
).reset_index(drop=True)

INTENT_LABELED_OUT = Path(
    "/kaggle/working/data/processed/Cleaned data/intent_labeled.csv"
)

INTENT_LABELED_OUT.parent.mkdir(
    parents=True,
    exist_ok=True
)

intent_labeled_df.to_csv(
    INTENT_LABELED_OUT,
    index=False
)

print("=== Final Intent Labeled Dataset ===")
print(f"Minimum confidence: {MIN_CONFIDENCE}")
print(f"Total labeled emails: {len(intent_labeled_df):,}")
print(f"Saved to: {INTENT_LABELED_OUT}")

print("\n=== Label Counts ===")
print(
    intent_labeled_df["intent_name"].value_counts()
)

print("\n=== Label Percentages ===")
print(
    (intent_labeled_df["intent_name"].value_counts(normalize=True) * 100)
    .round(2)
)

print("\n=== Missing Values ===")
print(
    intent_labeled_df[
        ["file", "cleaned_body", "intent_label", "intent_name"]
    ].isnull().sum()
)

print("\n=== Duplicate Files ===")
print(
    intent_labeled_df["file"].duplicated().sum()
)

print("\nFirst 5 labeled emails:")
display(
    intent_labeled_df[
        [
            "file",
            "subject",
            "intent_label",
            "intent_name",
            "confidence_score"
        ]
    ].head()
)

## Last verification cell


In [ ]:
print("=== Intent Labeled Verification ===")

print(f"Rows: {len(intent_labeled_df):,}")
print(f"Output exists: {INTENT_LABELED_OUT.exists()}")

print("\n=== Label Distribution ===")
print(
    intent_labeled_df["intent_name"].value_counts()
)

print("\n=== Missing Values ===")
print(
    intent_labeled_df[
        ["file", "cleaned_body", "intent_label", "intent_name"]
    ].isnull().sum()
)

print("\n=== Duplicate Files ===")
print(
    intent_labeled_df["file"].duplicated().sum()
)

## Extract the "data" section and explore its structure

In [ ]:
data = emailsum_data["data"]

print("Type of data:", type(data))

if isinstance(data, list):
    print("Number of records:", len(data))
    print("\nFirst record:")
    print(data[0])

elif isinstance(data, dict):
    print("Keys inside data:")
    print(data.keys())

## Splitting the intent_labeled.csv into train, test and val partitions

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

INTENT_SOURCE = Path("/kaggle/working/data/processed/Cleaned data/intent_labeled.csv")
INTENT_SPLIT_DIR = Path("/kaggle/working/data/processed/intent")

print("Intent source:")
print(INTENT_SOURCE)
print("Exists:", INTENT_SOURCE.exists())

if not INTENT_SOURCE.exists():
    raise FileNotFoundError(
        f"intent_labeled.csv not found at:\n{INTENT_SOURCE}\n"
        "Make sure the intent labeling cell was run and saved successfully."
    )

intent_df = pd.read_csv(INTENT_SOURCE)

print(f"\nTotal intent-labeled records: {len(intent_df):,}")

train_df, temp_df = train_test_split(
    intent_df,
    test_size=0.30,
    random_state=42,
    stratify=intent_df["intent_label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["intent_label"]
)

for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df)
]:
    split_dir = INTENT_SPLIT_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    output_path = split_dir / f"intent_{split_name}.csv"
    split_df.to_csv(output_path, index=False)

    print(f"{split_name}: {len(split_df):,} -> {output_path}")

print("\n=== Intent Split Verification ===")
print(f"Original: {len(intent_df):,}")
print(f"Train:    {len(train_df):,}")
print(f"Val:      {len(val_df):,}")
print(f"Test:     {len(test_df):,}")
print(f"Total:    {len(train_df) + len(val_df) + len(test_df):,}")
print(
    "Matches original:",
    len(train_df) + len(val_df) + len(test_df) == len(intent_df)
)

print("\n=== Intent Distribution ===")
print(
    pd.DataFrame({
        "train": train_df["intent_name"].value_counts(),
        "val": val_df["intent_name"].value_counts(),
        "test": test_df["intent_name"].value_counts()
    }).fillna(0).astype(int)
)

print("\n=== Created Files ===")
for path in sorted(INTENT_SPLIT_DIR.rglob("*.csv")):
    print(path)

## Creating a kaggle dataset from the created splits to be used as inputs

### Check what is there in the folder first

In [ ]:
!find /kaggle/working/data/processed/intent -maxdepth 2 -type f | head -50

### Create the dataset metadata

In [ ]:
import json

metadata = {
    "title": "processed-intent-data",
    "id": "YOUR_KAGGLE_USERNAME/processed-intent-data",
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ]
}

with open(
    "/kaggle/working/data/processed/intent/dataset-metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)

### Create the kaggle dataset

In [ ]:
!python -c "import os; print(os.environ.get('KAGGLE_USERNAME'))"

## Print the first three records in the train and test sets to understand data organization pattern

In [ ]:
print("\nFirst 3 training records:")
for i, record in enumerate(data["train"][:3]):
    print(f"\n--- Record {i} ---")
    print(record)

print("\nFirst 3 test records:")
for i, record in enumerate(data["test"][:3]):
    print(f"\n--- Record {i} ---")
    print(record['long_summary']['content'])

## Iterate through every in the 'data' section and print what all is present in it

In [ ]:
for key, value in data.items():
    print("\nKEY:", key)
    print("TYPE:", type(value))

    if isinstance(value, list):
        print("NUMBER OF ITEMS:", len(value))
        if value:
            print("FIRST ITEM:", value[0])
    elif isinstance(value, dict):
        print("SUBKEYS:", list(value.keys())[:20])

In [ ]:
# ============================================================
# CHECK PROCESSED DATASETS
# ============================================================

from pathlib import Path
import pandas as pd

print("=== Processed Dataset Status ===\n")

# Expected processed files
processed_files = {
    "enron_cleaned": CLEAN_DIR / "enron_cleaned.csv",
    "spamassassin_cleaned": CLEAN_DIR / "spamassassin_cleaned.csv",
    "intent_labeled": CLEAN_DIR / "intent_labeled.csv",

    "intent_train": TRAIN_DIR / "intent_train.csv",
    "intent_val": VAL_DIR / "intent_val.csv",
    "intent_test": TEST_DIR / "intent_test.csv",

    "enron_train": TRAIN_DIR / "enron_train.csv",
    "spam_train": TRAIN_DIR / "spamassassin_train.csv",
}

for name, path in processed_files.items():
    print(f"{name:25s} -> {path.exists()}")

# Data Preprocessing
Email Preprocessing Pipeline:
1. Parse raw email using Python's `email` library → extract headers and body separately
2. Split email thread into individual messages using delimiter patterns
3. Strip HTML tags from body using BeautifulSoup4
4. Remove email signatures (lines starting with "--", "Best", "Regards", "Thanks", etc.)
5. Remove quoted reply lines (lines starting with ">")
6. Normalize whitespace (collapse multiple blank lines)
7. Lowercase all text
8. Tokenize into sentences using spaCy or NLTK
9. Tokenize sentences into words
10. Remove stopwords using NLTK stopword list + custom email stopwords
11. Save cleaned output to CSV


## Preprocessing function library

In [ ]:
import os
import re
import time
import email
from email import policy
from pathlib import Path
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize

STOPWORDS = set(stopwords.words("english"))
EMAIL_STOPWORDS = {"fwd", "re", "fw", "subject", "from", "to", "cc", "bcc", "sent"}
ALL_STOPWORDS = STOPWORDS | EMAIL_STOPWORDS

SIGNATURE_MARKERS = (
    "--", "best,", "best regards", "regards,", "regards", "thanks,", "thanks",
    "thank you,", "sincerely,", "sincerely", "cheers,", "cheers", "sent from my",
)
QUOTE_PREFIXES = (">", "&gt;")


def parse_raw_email(raw_text):
    """Step 1 — parse a raw RFC822 message string (the `message` column of emails.csv,
    or a raw .eml file's contents) into headers + body via Python's email lib."""
    msg = email.message_from_string(raw_text, policy=policy.default)
    headers = {
        "from": msg.get("From", ""),
        "to": msg.get("To", ""),
        "subject": msg.get("Subject", ""),
        "date": msg.get("Date", ""),
    }
    body = ""
    if msg.is_multipart():
        for part in msg.walk():
            ctype = part.get_content_type()
            if ctype in ("text/plain", "text/html") and not part.get("Content-Disposition"):
                try:
                    body += part.get_content()
                except Exception:
                    pass
    else:
        try:
            body = msg.get_content()
        except Exception:
            body = str(msg.get_payload())
    return headers, body


def split_thread_into_messages(body):
    """Step 2 — split a reply chain into individual messages using common delimiter patterns."""
    delimiters = [
        r"-{2,}\s*Original Message\s*-{2,}",
        r"On .{5,80} wrote:",
        r"From:.{0,120}\nSent:.{0,120}\nTo:.{0,120}\nSubject:",
    ]
    parts = re.split("|".join(delimiters), body, flags=re.IGNORECASE)
    return [p.strip() for p in parts if p.strip()]


def strip_html(text):
    """Step 3 — strip HTML tags with BeautifulSoup4 (only runs BS4 when tags are present)."""
    if "<" in text and ">" in text:
        return BeautifulSoup(text, "html.parser").get_text(separator=" ")
    return text


def remove_signature(text):
    """Step 4 — cut everything from the first signature marker line onward."""
    lines = text.split("\n")
    cut_idx = len(lines)
    for i, line in enumerate(lines):
        stripped = line.strip().lower()
        if stripped in SIGNATURE_MARKERS or stripped.startswith("--"):
            cut_idx = i
            break
    return "\n".join(lines[:cut_idx])


def remove_quoted_lines(text):
    """Step 5 — drop quoted reply lines (start with '>')."""
    return "\n".join(l for l in text.split("\n") if not l.strip().startswith(QUOTE_PREFIXES))


def normalize_whitespace(text):
    """Step 6 — collapse repeated spaces/blank lines."""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def tokenize_sentences(text):
    """Step 8 — sentence tokenization (NLTK)."""
    return sent_tokenize(text)


def tokenize_words_no_stopwords(text):
    """Steps 9–10 — word tokenize + remove stopwords."""
    words = word_tokenize(text.lower())
    return [w for w in words if w.isalpha() and w not in ALL_STOPWORDS]


def clean_email_body(raw_body):
    """Full pipeline (steps 3–7) — returns the cleaned_body text used in the output CSVs."""
    text = strip_html(raw_body)
    text = remove_signature(text)
    text = remove_quoted_lines(text)
    text = normalize_whitespace(text)
    text = text.lower()  # step 7
    return text


print("Preprocessing functions ready: parse_raw_email, split_thread_into_messages,")
print("strip_html, remove_signature, remove_quoted_lines, normalize_whitespace,")
print("tokenize_sentences, tokenize_words_no_stopwords, clean_email_body")


## Test the preprocessing pipeline on 10 sample mails
Check that there are no HTML tags, '>' quoted lines and trailing signature blocks after pre-processing

In [ ]:
import random

ENRON_CSV = Path("/kaggle/input/datasets/wcukierski/enron-email-dataset/emails.csv")

emails_df = pd.read_csv(ENRON_CSV)
print(f"Total rows in emails.csv: {len(emails_df):,}")
print(f"Columns: {list(emails_df.columns)}")

random.seed(42)
sample_idx = random.sample(range(len(emails_df)), min(10, len(emails_df)))

for idx in sample_idx:
    row = emails_df.iloc[idx]
    raw = row["message"]
    headers, body = parse_raw_email(raw)
    cleaned = clean_email_body(body)

    print("=" * 70)
    print(f"File ID: {row['file']}")
    print(f"Subject: {headers['subject']}")
    print(f"--- RAW (first 300 chars) ---\n{body[:300]}")
    print(f"--- CLEANED (first 300 chars) ---\n{cleaned[:300]}")


## Preprocess the entire Enron dataset

In [ ]:
import time
import pandas as pd
from pathlib import Path


def process_enron_dataset(
    emails_csv_path,
    out_csv_path,
    limit=None,
    progress_every=10_000
):
    """
    Reads emails.csv (columns: file, message) and applies the
    Step 6.2 cleaning pipeline to the raw text in `message`.

    Parameters
    ----------
    emails_csv_path : Path
        Path to Enron emails.csv.
    out_csv_path : Path
        Output path for enron_cleaned.csv.
    limit : int or None
        Number of rows to process. None = entire dataset.
    progress_every : int
        Print progress every N processed rows.
    """

    start = time.time()

    # ---------------------------------------------------------
    # Load raw dataset
    # ---------------------------------------------------------

    print("=" * 70)
    print("STARTING ENRON PREPROCESSING")
    print("=" * 70)

    print(f"Input file : {emails_csv_path}")
    print(f"Output file: {out_csv_path}")

    raw_df = pd.read_csv(emails_csv_path)

    total_rows = len(raw_df)

    if limit is not None:
        raw_df = raw_df.head(limit)

    total_to_process = len(raw_df)

    print(f"\nTotal rows in source CSV : {total_rows:,}")
    print(f"Rows to process          : {total_to_process:,}")

    if limit is not None:
        print(f"Limit applied            : {limit:,}")
    else:
        print("Limit applied            : None (full dataset)")

    print("=" * 70)

    # ---------------------------------------------------------
    # Counters
    # ---------------------------------------------------------

    records = []

    skipped = 0
    empty = 0
    processed = 0

    # ---------------------------------------------------------
    # Process emails
    # ---------------------------------------------------------

    for _, row in raw_df.iterrows():

        try:
            headers, body = parse_raw_email(row["message"])

            cleaned = clean_email_body(body)

            # Skip empty / extremely short emails
            if not cleaned or len(cleaned.split()) < 3:
                empty += 1
                processed += 1

                if processed % progress_every == 0:
                    elapsed = time.time() - start
                    percent = (processed / total_to_process) * 100
                    rate = processed / elapsed if elapsed > 0 else 0
                    remaining = (
                        total_to_process - processed
                    )

                    print(
                        f"[{processed:,}/{total_to_process:,}] "
                        f"{percent:6.2f}% | "
                        f"Valid: {len(records):,} | "
                        f"Short/empty: {empty:,} | "
                        f"Errors: {skipped:,} | "
                        f"Rate: {rate:,.0f} rows/s | "
                        f"Elapsed: {elapsed/60:.1f} min"
                    )

                continue

            # Store cleaned record
            records.append({
                "file": row["file"],
                "sender": headers["from"],
                "date": headers["date"],
                "subject": headers["subject"],
                "cleaned_body": cleaned,
            })

        except Exception:
            skipped += 1

        processed += 1

        # -----------------------------------------------------
        # Progress report
        # -----------------------------------------------------

        if processed % progress_every == 0:

            elapsed = time.time() - start

            percent = (
                processed / total_to_process
            ) * 100

            rate = (
                processed / elapsed
                if elapsed > 0
                else 0
            )

            remaining = total_to_process - processed

            eta_seconds = (
                remaining / rate
                if rate > 0
                else 0
            )

            print(
                f"[{processed:,}/{total_to_process:,}] "
                f"{percent:6.2f}% | "
                f"Valid: {len(records):,} | "
                f"Short/empty: {empty:,} | "
                f"Errors: {skipped:,} | "
                f"Rate: {rate:,.0f} rows/s | "
                f"ETA: {eta_seconds/60:.1f} min"
            )

    # ---------------------------------------------------------
    # Create output DataFrame
    # ---------------------------------------------------------

    df = pd.DataFrame(records)

    # Make sure output directory exists
    out_csv_path = Path(out_csv_path)
    out_csv_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Save
    df.to_csv(
        out_csv_path,
        index=False
    )

    # ---------------------------------------------------------
    # Final statistics
    # ---------------------------------------------------------

    elapsed = time.time() - start

    print("\n")
    print("=" * 70)
    print("ENRON PREPROCESSING COMPLETE")
    print("=" * 70)

    print(f"Source rows          : {total_rows:,}")
    print(f"Rows processed       : {processed:,}")
    print(f"Valid cleaned rows   : {len(df):,}")
    print(f"Empty/too short      : {empty:,}")
    print(f"Skipped due to error : {skipped:,}")

    print(
        f"Success rate         : "
        f"{(len(df) / processed * 100):.2f}%"
    )

    print(
        f"Total elapsed time   : "
        f"{elapsed / 60:.2f} minutes"
    )

    print(
        f"Average speed        : "
        f"{processed / elapsed:,.0f} rows/sec"
    )

    print(f"Saved to             : {out_csv_path}")
    print(f"File exists          : {out_csv_path.exists()}")

    print("=" * 70)

    return df


# ============================================================
# FULL ENRON PREPROCESSING
# ============================================================

ENRON_CSV = Path(
    "/kaggle/input/datasets/wcukierski/enron-email-dataset/emails.csv"
)

# Use the output directory from your NEW config cell
ENRON_OUT = CLEAN_DIR / "enron_cleaned.csv"


enron_cleaned_df = process_enron_dataset(
    ENRON_CSV,
    ENRON_OUT,
    limit=None,
    progress_every=10_000
)

print("\nFirst 5 cleaned records:")
display(enron_cleaned_df.head())

## Preprocessing the SpamAssassin Dataset

In [ ]:
# ============================================================
# STEP 6.5 — PROCESS SPAMASSASSIN
# ============================================================

SPAM_RAW_DIR = Path(
    "/kaggle/input/datasets/prishabhkumar/"
    "spamassassin/kaggle/working/spamassassin_raw"
)

SPAM_FOLDER_LABELS = {
    "easy_ham": 0,
    "hard_ham": 0,
    "spam": 1,
    "spam_2": 1,
}


def process_spamassassin_dataset(
    raw_dir,
    out_csv_path,
    progress_every=1000
):
    records = []
    skipped = 0
    processed = 0

    start = time.time()

    print("=" * 70)
    print("STARTING SPAMASSASSIN PREPROCESSING")
    print("=" * 70)

    print(f"Input directory : {raw_dir}")
    print(f"Output file     : {out_csv_path}")

    # --------------------------------------------------------
    # Check folders
    # --------------------------------------------------------

    print("\nChecking source folders:")

    for folder_name in SPAM_FOLDER_LABELS:

        folder_path = raw_dir / folder_name

        print(
            f"  {folder_name:10s} -> "
            f"{'FOUND' if folder_path.exists() else 'MISSING'}"
        )

    # --------------------------------------------------------
    # Process folders
    # --------------------------------------------------------

    for folder_name, label in SPAM_FOLDER_LABELS.items():

        folder_path = raw_dir / folder_name

        if not folder_path.exists():
            print(f"\nSkipping missing folder: {folder_name}")
            continue

        files = [
            f for f in folder_path.iterdir()
            if f.is_file()
        ]

        print(
            f"\nProcessing {folder_name}: "
            f"{len(files):,} files"
        )

        for fpath in files:

            try:
                with open(
                    fpath,
                    "r",
                    encoding="latin-1"
                ) as fh:
                    raw = fh.read()

                headers, body = parse_raw_email(raw)

                cleaned = clean_email_body(body)

                if not cleaned:
                    processed += 1
                    continue

                records.append({
                    "file": f"{folder_name}/{fpath.name}",
                    "cleaned_body": cleaned,
                    "label": label,
                })

            except Exception:
                skipped += 1

            processed += 1

            # ------------------------------------------------
            # Progress
            # ------------------------------------------------

            if processed % progress_every == 0:

                elapsed = time.time() - start

                rate = (
                    processed / elapsed
                    if elapsed > 0
                    else 0
                )

                print(
                    f"  Progress: {processed:,} | "
                    f"Valid: {len(records):,} | "
                    f"Skipped: {skipped:,} | "
                    f"Rate: {rate:,.0f} files/sec | "
                    f"Elapsed: {elapsed/60:.2f} min"
                )

   

    if not records:
        raise RuntimeError(
            "\nNo emails were processed.\n"
            "Please verify the SpamAssassin folder structure."
        )

   

    df = pd.DataFrame(records)

    out_csv_path = Path(out_csv_path)

    out_csv_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        out_csv_path,
        index=False
    )

    

    elapsed = time.time() - start

    print("\n")
    print("=" * 70)
    print("SPAMASSASSIN PREPROCESSING COMPLETE")
    print("=" * 70)

    print(f"Files processed : {processed:,}")
    print(f"Valid records   : {len(df):,}")
    print(f"Skipped/errors  : {skipped:,}")

    print("\nLabel distribution:")

    print(
        df["label"]
        .value_counts()
        .sort_index()
        .rename({
            0: "ham",
            1: "spam"
        })
    )

    print(f"\nElapsed time    : {elapsed / 60:.2f} minutes")
    print(f"Saved to        : {out_csv_path}")
    print(f"File exists     : {out_csv_path.exists()}")

    print("=" * 70)

    return df




SPAM_OUT = CLEAN_DIR / "spamassassin_cleaned.csv"




spam_cleaned_df = process_spamassassin_dataset(
    SPAM_RAW_DIR,
    SPAM_OUT,
    progress_every=1000
)

display(spam_cleaned_df.head())

## Preprocessing the BC3 corpus
This is the updated preprocessing cell after checking the folder structure and merging the files based on the matching keys

In [ ]:
import xml.etree.ElementTree as ET
import json
from pathlib import Path

BC3_DIR = Path("/kaggle/input/datasets/prishabhkumar/bc3-corpus")

BC3_CORPUS_XML = BC3_DIR / "BC3" / "corpus.xml"
BC3_ANNOTATION_XML = BC3_DIR / "BC3" / "annotation.xml"

BC3_OUT = WORK_DIR / "data/processed/Cleaned data/bc3_threads.json"


def process_bc3_dataset(corpus_xml, annotation_xml, out_json_path):

    corpus_root = ET.parse(corpus_xml).getroot()
    annotation_root = ET.parse(annotation_xml).getroot()

    annotation_map = {}

    for thread in annotation_root.findall(".//thread"):
        name = thread.findtext("name", default="").strip()

        summaries = []

        for summary in thread.findall(".//summary"):
            text = " ".join(
                s.text.strip()
                for s in summary.findall(".//sent")
                if s.text and s.text.strip()
            )

            if text:
                summaries.append(text)

        annotation_map[name] = summaries

    threads = []
    unmatched = 0

    for thread in corpus_root.findall(".//thread"):

        name = thread.findtext("name", default="").strip()
        listno = thread.findtext("listno", default="").strip()

        if name not in annotation_map:
            unmatched += 1
            continue

        messages = []

        for doc in thread.findall("DOC"):

            sender = doc.findtext("From", default="").strip()
            subject = doc.findtext("Subject", default="").strip()

            body = " ".join(
                s.text.strip()
                for s in doc.findall(".//Sent")
                if s.text and s.text.strip()
            )

            cleaned_body = clean_email_body(body)

            if cleaned_body:
                messages.append({
                    "sender": sender,
                    "subject": subject,
                    "body": cleaned_body
                })

        threads.append({
            "thread_id": listno,
            "thread_name": name,
            "messages": messages,
            "human_summaries": annotation_map[name]
        })

    out_json_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_json_path, "w", encoding="utf-8") as f:
        json.dump(
            threads,
            f,
            indent=2,
            ensure_ascii=False
        )

    total_messages = sum(len(t["messages"]) for t in threads)
    total_summaries = sum(len(t["human_summaries"]) for t in threads)

    print(f"Threads: {len(threads):,}")
    print(f"Messages: {total_messages:,}")
    print(f"Summaries: {total_summaries:,}")
    print(f"Unmatched corpus threads: {unmatched:,}")
    print(f"Saved to: {out_json_path}")

    return threads


bc3_threads = process_bc3_dataset(
    BC3_CORPUS_XML,
    BC3_ANNOTATION_XML,
    BC3_OUT
)

## Finding the matching key
There are two files in the BC3 folder and also they are not aligned by poisition hence we find the matching keys in both the files

In [ ]:
corpus_threads = corpus_root.findall(".//thread")
annotation_threads = annotation_root.findall(".//thread")

print("Corpus threads:", len(corpus_threads))
print("Annotation threads:", len(annotation_threads))

corpus_listnos = {
    t.findtext("listno", default="").strip()
    for t in corpus_threads
}

annotation_listnos = {
    t.findtext("listno", default="").strip()
    for t in annotation_threads
}

print("\nMatching listno values:",
      len(corpus_listnos & annotation_listnos))

print("Corpus-only listnos:",
      len(corpus_listnos - annotation_listnos))

print("Annotation-only listnos:",
      len(annotation_listnos - corpus_listnos))

In [ ]:
corpus_names = {
    t.findtext("name", default="").strip()
    for t in corpus_threads
}

annotation_names = {
    t.findtext("name", default="").strip()
    for t in annotation_threads
}

matches = corpus_names & annotation_names

print("Matching thread names:", len(matches))

print("\nSome matching names:")
for name in list(matches)[:10]:
    print("-", name)

## Creating the Train/Test/Val split

In [ ]:
from sklearn.model_selection import train_test_split

def make_splits(df, out_dir, name, test_size=0.15, val_size=0.18, stratify_col=None, seed=42):
    """70% train / ~15% val / 15% test. Val is carved out of the remainder after the
    test split, so val_size=0.18 of the 85% remainder works out to ~15% of the total."""
    stratify = df[stratify_col] if stratify_col else None
    train_df, test_df = train_test_split(df, test_size=test_size, random_state=seed, stratify=stratify)

    stratify2 = train_df[stratify_col] if stratify_col else None
    train_df, val_df = train_test_split(train_df, test_size=val_size, random_state=seed, stratify=stratify2)

    for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        out_path = out_dir / split_name / f"{name}_{split_name}.csv"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        split_df.to_csv(out_path, index=False)

    total = len(train_df) + len(val_df) + len(test_df)
    print(f"{name}: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}, "
          f"sum={total}, matches original={total == len(df)}")
    return train_df, val_df, test_df

PROCESSED_DIR = Path("data/processed")

enron_train, enron_val, enron_test = make_splits(enron_cleaned_df, PROCESSED_DIR, "enron")
spam_train, spam_val, spam_test = make_splits(spam_cleaned_df, PROCESSED_DIR, "spamassassin", stratify_col="label")

# The intent split (intent_train/val/test.csv) is created in Phase 7, Step 7.5, once
# intent_labeled.csv exists — it reuses this same make_splits() helper with
# stratify_col="intent_label" so each class is represented in every split.


## Creating the folder for the output data folder

In [ ]:
from pathlib import Path
import shutil

SOURCE_DIR = Path("/kaggle/working/data/processed")
DATASET_DIR = Path("/kaggle/working/email_intelligence_processed_data")

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

shutil.copytree(SOURCE_DIR, DATASET_DIR)

print("Prepared dataset folder:")
for path in sorted(DATASET_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(DATASET_DIR))

## Data preparation for summarization

### Understand the structure in which the threads are stored

In [ ]:
from pathlib import Path
import json

EMAILSUM_DIR = Path("/kaggle/input/datasets/prishabhkumar/emailsum")

MAIN_SUMMARY_JSON = (
    EMAILSUM_DIR / "Avocado" / "summaries" / "EmailSum_data.json"
)

EXTRA_REF_JSON = (
    EMAILSUM_DIR / "Avocado" / "summaries" / "one_more_reference.json"
)

with open(MAIN_SUMMARY_JSON) as f:
    emailsum_summaries = json.load(f)["data"]["train"]

with open(EXTRA_REF_JSON) as f:
    emailsum_extra_ref = json.load(f)["test"]

print(f"EmailSum_data.json: {len(emailsum_summaries)} threads")
print(f"one_more_reference.json: {len(emailsum_extra_ref)} threads")

for h in emailsum_summaries[:1]:
    print(json.dumps(h, indent=2))

### Understand the structure of the second file (one_more_reference.json) as well

In [ ]:
with open(EXTRA_REF_JSON) as f:
    extra_ref_check = json.load(f)

print(type(extra_ref_check))

if isinstance(extra_ref_check, dict):
    print("Top-level keys:")
    print(extra_ref_check.keys())
else:
    print("Number of items:", len(extra_ref_check))

### Loading the summary files of EMAILSUM and check the number of threads present in both files

In [ ]:
with open(MAIN_SUMMARY_JSON) as f:
    emailsum_summaries = json.load(f)["data"]["train"]

with open(EXTRA_REF_JSON) as f:
    emailsum_extra_ref = json.load(f)["test"]

print(f"EmailSum_data.json: {len(emailsum_summaries)} threads")
print(f"one_more_reference.json: {len(emailsum_extra_ref)} threads")

for h in emailsum_summaries[:1]:
    print(json.dumps(h, indent=2))

### Loading/inspecting the BC3 dataset

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

BC3_XML = Path(
    "/kaggle/input/datasets/prishabhkumar/bc3-corpus/BC3/corpus.xml"
)

print("BC3 file exists:", BC3_XML.exists())
print("BC3 file:", BC3_XML)

# Parse the XML
tree = ET.parse(BC3_XML)
root = tree.getroot()

print("\nRoot tag:", root.tag)
print("Root attributes:", root.attrib)

# Show the first few levels/elements
print("\nFirst-level elements:")
for child in root:
    print(" -", child.tag, child.attrib)

### Inspect content inside every thread

In [ ]:
# Inspect the first BC3 thread in detail

first_thread = root.find("thread")

print("Thread tag:", first_thread.tag)
print("Thread attributes:", first_thread.attrib)

print("\nChildren of first thread:")
for child in first_thread:
    print(" -", child.tag, child.attrib)

print("\nFirst thread XML:")
print(ET.tostring(first_thread, encoding="unicode")[:5000])

### Inspect other files in the folder to find the location of the actual sumaries

In [ ]:
from pathlib import Path

BC3_DIR = Path("/kaggle/input/datasets/prishabhkumar/bc3-corpus")

print("BC3 dataset contents:\n")

for path in BC3_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(BC3_DIR))

### Now, we are inspecting the annotations.xml because summaries are present there and the corpus.xml has the actual email threads

In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET

BC3_ANNOTATION_XML = Path(
    "/kaggle/input/datasets/prishabhkumar/bc3-corpus/BC3/annotation.xml"
)

print("Annotation file exists:", BC3_ANNOTATION_XML.exists())
print("Annotation file:", BC3_ANNOTATION_XML)

# Parse the XML
annotation_tree = ET.parse(BC3_ANNOTATION_XML)
annotation_root = annotation_tree.getroot()

print("\nRoot tag:", annotation_root.tag)
print("Root attributes:", annotation_root.attrib)

print("\nFirst-level elements:")
for child in annotation_root:
    print(" -", child.tag, child.attrib)

print("\nXML preview:")
print(
    ET.tostring(annotation_root, encoding="unicode")[:5000]
)

### Check if corpus.xml and annotations.xml contain the same matching threads

In [ ]:
# Verify that corpus.xml and annotation.xml contain matching threads

corpus_threads = {
    thread.findtext("listno")
    for thread in root.findall("thread")
}

annotation_threads = {
    thread.findtext("listno")
    for thread in annotation_root.findall("thread")
}

print("Corpus threads:", len(corpus_threads))
print("Annotation threads:", len(annotation_threads))

print("\nThreads in corpus but not annotation:",
      len(corpus_threads - annotation_threads))

print("Threads in annotation but not corpus:",
      len(annotation_threads - corpus_threads))

print("\nMatching threads:",
      len(corpus_threads & annotation_threads))

### Identification of the mismatching threads

In [ ]:
# Identify the unmatched BC3 threads

corpus_only = corpus_threads - annotation_threads
annotation_only = annotation_threads - corpus_threads

print("Thread present only in corpus.xml:")
print(corpus_only)

print("\nThread present only in annotation.xml:")
print(annotation_only)

### Check the level of matching after normalizing the names

In [ ]:
# Normalize annotation thread IDs for matching.
# The annotation file has one thread ID with a trailing '+'
# that is not present in corpus.xml.

normalized_annotation_threads = {
    thread.findtext("listno").rstrip("+")
    for thread in annotation_root.findall("thread")
}

print("Corpus threads:", len(corpus_threads))
print("Normalized annotation threads:", len(normalized_annotation_threads))

print("\nThreads in corpus but not annotation:")
print(corpus_threads - normalized_annotation_threads)

print("\nThreads in annotation but not corpus:")
print(normalized_annotation_threads - corpus_threads)

print("\nMatching threads:")
print(len(corpus_threads & normalized_annotation_threads))

### Reformatting the BC3 data into : [input, short_summary]

In [ ]:
# Phase 8.3 — Reformat BC3 into {input, short_summary}

bc3_records = []

# Create a lookup for the annotation threads.
# Normalize the trailing '+' in the annotation list number.
annotation_lookup = {}

for thread in annotation_root.findall("thread"):
    listno = thread.findtext("listno").rstrip("+")
    
    # Collect all summary sentences from the annotations.
    summaries = []

    for annotation in thread.findall("annotation"):
        summary = annotation.find("summary")

        if summary is not None:
            for sent in summary.findall("sent"):
                text = (sent.text or "").strip()

                if text:
                    summaries.append(text)

    # Keep the summaries associated with this thread.
    annotation_lookup[listno] = summaries


# Extract the actual email text from corpus.xml
for thread in root.findall("thread"):
    listno = thread.findtext("listno")

    # Find the corresponding annotation.
    summaries = annotation_lookup.get(listno, [])

    if not summaries:
        continue

    # Collect all sentences from all emails in the thread.
    input_sentences = []

    for doc in thread.findall("DOC"):
        text = doc.find("Text")

        if text is not None:
            for sent in text.findall("Sent"):
                sentence = (sent.text or "").strip()

                if sentence:
                    input_sentences.append(sentence)

    # Combine the complete thread into one input.
    thread_input = " ".join(input_sentences)

    # Combine the reference summary sentences.
    short_summary = " ".join(summaries)

    if thread_input and short_summary:
        bc3_records.append({
            "input": thread_input,
            "short_summary": short_summary
        })


print(f"BC3 records created: {len(bc3_records)}")

print("\nFirst BC3 record:")
print(bc3_records[0])

## Back to preparation of EMAILSUM

### FInding the location of the file that contain the actual thread text

In [ ]:
print("EMAILSUM dataset contents:\n")

for path in EMAILSUM_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(EMAILSUM_DIR))

### Inspecting the thread-extraction script 
In the emailsum folder, we dont expicitly have something that gives us the email thread texts. However, we have a script called extract_threads.py and hence we need to check what it does because it might give us a way to accquire the threads

In [ ]:
EXTRACT_THREADS_PY = EMAILSUM_DIR / "Avocado" / "extact_threads.py"

with open(EXTRACT_THREADS_PY, "r", encoding="utf-8") as f:
    extract_threads_code = f.read()

print(extract_threads_code)

### Way to extract email threads using extract_threads.py
Now the extract_threads.py is the file that is used to extract the actual email threads and store it in a file called avocado.json. 

In [ ]:
from pathlib import Path

print("Searching Kaggle input for Avocado.json...\n")

matches = []

for path in Path("/kaggle/input").rglob("Avocado.json"):
    matches.append(path)

if matches:
    for path in matches:
        print(path)
else:
    print("Avocado.json was not found in /kaggle/input")

# Fallback dataset for summarization

### Since access to the LDCA artificat is not possible, we are moving ahead with a fallback 'Email Thread Summary Dataset' data source that is publicly avilable on kaggle

### Locate the data source and check whether it contains the required file

In [ ]:
from pathlib import Path
import pandas as pd
import json
import os

# Search Kaggle input for the fallback dataset files
candidates = list(Path("/kaggle/input").rglob("email_thread_details*"))

print("Candidate detail files:")
for p in candidates:
    print(" ", p)

if not candidates:
    raise FileNotFoundError(
        "Could not find email_thread_details. "
        "Make sure 'marawanxmamdouh/email-thread-summary-dataset' "
        "has been attached using Add Data."
    )

FALLBACK_DIR = candidates[0].parent

DETAILS_CSV = FALLBACK_DIR / "email_thread_details.csv"
SUMMARIES_CSV = FALLBACK_DIR / "email_thread_summaries.csv"

print("\nFallback dataset directory:")
print(FALLBACK_DIR)

print("\nFiles:")
for f in sorted(FALLBACK_DIR.rglob("*")):
    if f.is_file():
        print(" ", f)

print("\nDetails CSV exists:", DETAILS_CSV.exists())
print("Summaries CSV exists:", SUMMARIES_CSV.exists())

### Load and analyze the structure of both of the above files

In [ ]:
details_df = pd.read_csv(DETAILS_CSV)
summaries_df = pd.read_csv(SUMMARIES_CSV)

print("========== DETAILS ==========")
print("Shape:", details_df.shape)
print("Columns:", details_df.columns.tolist())
print(details_df.head(3))

print("\n========== SUMMARIES ==========")
print("Shape:", summaries_df.shape)
print("Columns:", summaries_df.columns.tolist())
print(summaries_df.head(3))

print("\n========== THREAD COUNTS ==========")
print("Unique detail threads:", details_df["thread_id"].nunique())
print("Unique summary threads:", summaries_df["thread_id"].nunique())

print("\n========== MISSING VALUES ==========")
print(details_df.isna().sum())
print()
print(summaries_df.isna().sum())

### Grouping the mails and re-constructing the threads
Here, we are gourping all the 21684 mails which are there based on the thread_id to get proper email threads that can bse used for summarization

In [ ]:
# Create a copy to work on so that the oringally loaded dataframes stay as it is
details = details_df.copy()
summaries = summaries_df.copy()

# Make sure that the thread IDs are of the same type in both the files
details["thread_id"] = details["thread_id"].astype(str)
summaries["thread_id"] = summaries["thread_id"].astype(str)

# Right now we have the timestamp which is in a different format hence we need to convert it into date time
details["timestamp_parsed"] = pd.to_datetime(
    details["timestamp"],
    errors="coerce"
)

# sorting every email in a single thread in chronological order
details = details.sort_values(
    ["thread_id", "timestamp_parsed"],
    kind="stable"
)

# combine all the emails that belong to the same thread
def combine_thread(group):
    messages = []

    for _, row in group.iterrows():
        body = str(row["body"]).strip()

        if body:
            messages.append(body)

    # make sure that every single individual mail inside a thread has clear separation
    return "\n\n--- EMAIL MESSAGE ---\n\n".join(messages)


thread_df = (
    details
    .groupby("thread_id", sort=False)
    .apply(
        combine_thread,
        include_groups=False
    )
    .reset_index(name="email_body")
)

print("========== RECONSTRUCTED THREADS ==========")
print("Number of threads:", len(thread_df))
print("Columns:", thread_df.columns.tolist())

print("\nFirst reconstructed thread:")
print(thread_df.iloc[0]["email_body"][:3000])

### Matching every single thread with is corresponding summary

In [ ]:
# Keep only the columns we need from the summary file
summaries_for_merge = summaries[
    ["thread_id", "summary"]
].copy()

# Make sure that there is only one reference summary for every thread
summaries_for_merge = summaries_for_merge.drop_duplicates(
    subset=["thread_id"]
)

# Join reconstructed email threads with their reference summaries
summarization_df = thread_df.merge(
    summaries_for_merge,
    on="thread_id",
    how="inner"
)

print("========== MATCHING RESULT ==========")
print("Reconstructed threads:", len(thread_df))
print("Reference summaries:", len(summaries_for_merge))
print("Matched thread-summary pairs:", len(summarization_df))

print("\nColumns:")
print(summarization_df.columns.tolist())

print("\nMissing values:")
print(summarization_df[["email_body", "summary"]].isna().sum())

print("\n========== SAMPLE MATCH ==========")
print("Thread ID:", summarization_df.iloc[0]["thread_id"])

print("\nEMAIL THREAD:")
print(summarization_df.iloc[0]["email_body"][:3000])

print("\nREFERENCE SUMMARY:")
print(summarization_df.iloc[0]["summary"])

### Saving the thread-summary combined dataframe before creating the split

In [ ]:
from pathlib import Path

SUMMARIZATION_DATA_DIR = Path(
    "/kaggle/working/data/processed/email_thread_summary"
)

SUMMARIZATION_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MASTER_SUMMARIZATION_FILE = (
    SUMMARIZATION_DATA_DIR / "email_thread_summary_all.csv"
)

summarization_df.to_csv(
    MASTER_SUMMARIZATION_FILE,
    index=False
)

print("Saved master summarization dataset:")
print(MASTER_SUMMARIZATION_FILE)

print("\nRows:", len(summarization_df))
print("Columns:", summarization_df.columns.tolist())

print(
    "File size:",
    round(MASTER_SUMMARIZATION_FILE.stat().st_size / (1024 * 1024), 2),
    "MB"
)

### Checking the lengts of the threads and the summaries to adjust the T5 configurations 
The reason we need to check the thread and summary length is that because we are having a size of 36.4 MB for just around 4167 threads which indicates that some threads might be very long which might cause issues when fed into the T5

In [ ]:
analysis_df = summarization_df.copy()

analysis_df["thread_words"] = (
    analysis_df["email_body"]
    .astype(str)
    .str.split()
    .str.len()
)

analysis_df["summary_words"] = (
    analysis_df["summary"]
    .astype(str)
    .str.split()
    .str.len()
)

print("THREAD LENGTHS")
print(analysis_df["thread_words"].describe())

print("\nSUMMARY LENGTHS")
print(analysis_df["summary_words"].describe())

print("\nLongest thread words:",
      analysis_df["thread_words"].max())

print("Longest summary words:",
      analysis_df["summary_words"].max())

### Check the effect of the quality filters mentioned in the guide

In [ ]:
print("========== ORIGINAL DATASET ==========")
print("Total pairs:", len(summarization_df))

# Same criteria used in the master implementation guide
input_words = summarization_df["email_body"].astype(str).str.split().str.len()
summary_words = summarization_df["summary"].astype(str).str.split().str.len()

valid_by_guide = (
    (input_words >= 50) &
    (summary_words <= 150) &
    (summary_words > 0)
)

print("\n========== GUIDE FILTER ==========")
print("Pairs passing filter:", valid_by_guide.sum())
print("Pairs removed:", (~valid_by_guide).sum())

print(
    "Percentage retained:",
    f"{valid_by_guide.mean():.2%}"
)

print(
    "Summaries >150 words:",
    (summary_words > 150).sum()
)

print(
    "Threads <50 words:",
    (input_words < 50).sum()
)

### Apply the quality filters and create the splits

In [ ]:
# ============================================================
# CELL 18E — Apply quality filter and create train/val/test
# ============================================================

# Work from the complete matched dataset
model_df = summarization_df.copy()

# Calculate word counts
model_df["input_words"] = (
    model_df["email_body"]
    .astype(str)
    .str.split()
    .str.len()
)

model_df["target_words"] = (
    model_df["summary"]
    .astype(str)
    .str.split()
    .str.len()
)

# Apply the same quality criteria used in the master guide
model_df = model_df[
    (model_df["input_words"] >= 50) &
    (model_df["target_words"] <= 150) &
    (model_df["target_words"] > 0)
].copy()

# Remove helper columns
model_df = model_df[
    ["thread_id", "email_body", "summary"]
].reset_index(drop=True)

print("========== FILTERED DATASET ==========")
print("Original pairs:", len(summarization_df))
print("Filtered pairs:", len(model_df))
print("Removed pairs:", len(summarization_df) - len(model_df))

# ------------------------------------------------------------
# Shuffle reproducibly
# ------------------------------------------------------------

model_df = model_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# ------------------------------------------------------------
# 70 / 15 / 15 split
# ------------------------------------------------------------

n = len(model_df)

train_end = int(n * 0.70)
val_end = train_end + int(n * 0.15)

emailsum_train_df = model_df.iloc[:train_end].copy()
emailsum_val_df = model_df.iloc[train_end:val_end].copy()
emailsum_test_df = model_df.iloc[val_end:].copy()

print("\n========== SPLIT RESULTS ==========")
print("Total:", len(model_df))
print("Train:", len(emailsum_train_df))
print("Validation:", len(emailsum_val_df))
print("Test:", len(emailsum_test_df))

# ------------------------------------------------------------
# Verify no thread appears in multiple splits
# ------------------------------------------------------------

train_ids = set(emailsum_train_df["thread_id"])
val_ids = set(emailsum_val_df["thread_id"])
test_ids = set(emailsum_test_df["thread_id"])

print("\n========== OVERLAP CHECK ==========")
print("Train ∩ Validation:", len(train_ids & val_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Validation ∩ Test:", len(val_ids & test_ids))

print(
    "Total unique thread IDs:",
    len(train_ids | val_ids | test_ids)
)

### Saving the splits into the working folder


In [ ]:
# ============================================================
# CELL 18F — Save master + train/validation/test datasets
# ============================================================

from pathlib import Path

BASE_DIR = Path("/kaggle/working/data/processed/email_thread_summary")

# Create split directories
for split in ["train", "val", "test"]:
    (BASE_DIR / split).mkdir(parents=True, exist_ok=True)

# File paths
MASTER_FILE = BASE_DIR / "email_thread_summary_all.csv"
TRAIN_FILE = BASE_DIR / "train" / "email_thread_summary_train.csv"
VAL_FILE = BASE_DIR / "val" / "email_thread_summary_val.csv"
TEST_FILE = BASE_DIR / "test" / "email_thread_summary_test.csv"

# Save master dataset
summarization_df.to_csv(
    MASTER_FILE,
    index=False
)

# Save filtered splits
emailsum_train_df.to_csv(
    TRAIN_FILE,
    index=False
)

emailsum_val_df.to_csv(
    VAL_FILE,
    index=False
)

emailsum_test_df.to_csv(
    TEST_FILE,
    index=False
)

# ============================================================
# Verification
# ============================================================

print("========== DATASET FILES SAVED ==========")

for path in [
    MASTER_FILE,
    TRAIN_FILE,
    VAL_FILE,
    TEST_FILE
]:
    print(
        f"{'OK' if path.exists() else 'MISSING'} | "
        f"{path} | "
        f"{path.stat().st_size / (1024 * 1024):.2f} MB"
        if path.exists()
        else f"MISSING | {path}"
    )

print("\n========== ROW COUNTS ==========")
print("Master:", len(summarization_df))
print("Train:", len(emailsum_train_df))
print("Validation:", len(emailsum_val_df))
print("Test:", len(emailsum_test_df))

print("\n========== DIRECTORY CONTENTS ==========")

for item in sorted(BASE_DIR.rglob("*")):
    print(
        "DIR  " if item.is_dir() else "FILE ",
        item.relative_to(BASE_DIR)
    )

## Creating the kaggle dataset from the saved data

### Initialize the kaggle dataset metadata

In [ ]:
!kaggle datasets init -p /kaggle/working/data/processed/email_thread_summary

### Create the metadata

In [ ]:
import json
from pathlib import Path

BASE_DIR = Path(
    "/kaggle/working/data/processed/email_thread_summary"
)

metadata = {
    "title": "Email Thread Summarization Dataset",
    "id": "prishabhkumar/email-thread-summarization-dataset",
    "subtitle": "Processed email threads with reference summaries",
    "description": (
        "Processed dataset for supervised email-thread summarization. "
        "Contains reconstructed email threads paired with reference summaries, "
        "along with train, validation, and test splits."
    ),
    "licenses": [
        {
            "name": "other"
        }
    ]
}

metadata_file = BASE_DIR / "dataset-metadata.json"

with open(metadata_file, "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved to:")
print(metadata_file)

print("\nMetadata:")
print(json.dumps(metadata, indent=2))

### Create the dataset

In [ ]:
!kaggle datasets create \
    -p /kaggle/working/data/processed/email_thread_summary \
    --dir-mode tar \
    --keep-tabular

## Creating the new staging directory to replace the current one


In [15]:
from pathlib import Path
import shutil

SOURCE_DIR = Path(
    "/kaggle/input/datasets/prishabhkumar/processed-intent-data"
)

STAGING_DIR = Path(
    "/kaggle/working/processed-intent-data"
)

NEW_FILE = Path(
    "/kaggle/working/data/processed/Cleaned data/intent_labeled.csv"
)

# Remove old staging directory if it already exists
if STAGING_DIR.exists():
    shutil.rmtree(STAGING_DIR)

# Create fresh staging directory
STAGING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Source:")
print(SOURCE_DIR)

print("\nStaging:")
print(STAGING_DIR)

print("\nNew file:")
print(NEW_FILE)

print("\nChecks:")
print("Source exists:", SOURCE_DIR.exists())
print("New file exists:", NEW_FILE.exists())

Source:
/kaggle/input/datasets/prishabhkumar/processed-intent-data

Staging:
/kaggle/working/processed-intent-data

New file:
/kaggle/working/data/processed/Cleaned data/intent_labeled.csv

Checks:
Source exists: True
New file exists: True


## Copy the existing dataset to the staging directory

In [16]:
import shutil

for item in SOURCE_DIR.iterdir():

    destination = STAGING_DIR / item.name

    if item.is_dir():
        shutil.copytree(
            item,
            destination
        )
    else:
        shutil.copy2(
            item,
            destination
        )

print("Existing dataset copied successfully.")

Existing dataset copied successfully.


## Add the new intent_lavel.csv to the staging directory

In [17]:
import shutil

shutil.copy2(
    NEW_FILE,
    STAGING_DIR / "intent_labeled.csv"
)

print("========== NEW FILE ADDED ==========")
print("Source:")
print(NEW_FILE)

print("\nDestination:")
print(STAGING_DIR / "intent_labeled.csv")

print("\nExists:")
print((STAGING_DIR / "intent_labeled.csv").exists())

========== NEW FILE ADDED ==========
Source:
/kaggle/working/data/processed/Cleaned data/intent_labeled.csv

Destination:
/kaggle/working/processed-intent-data/intent_labeled.csv

Exists:
True


## verify the entire structure before uploading

In [18]:
print("========== STAGING DATASET STRUCTURE ==========")

for path in sorted(STAGING_DIR.rglob("*")):

    relative_path = path.relative_to(STAGING_DIR)

    if path.is_dir():
        print(f"DIR  {relative_path}")
    else:
        size_mb = path.stat().st_size / (1024 * 1024)
        print(
            f"FILE {relative_path} | "
            f"{size_mb:.2f} MB"
        )

========== STAGING DATASET STRUCTURE ==========
FILE intent_labeled.csv | 143.44 MB
DIR  test
FILE test/intent_test.csv | 21.07 MB
DIR  train
FILE train/intent_train.csv | 101.21 MB
DIR  val
FILE val/intent_val.csv | 21.16 MB


In [19]:
import pandas as pd

files_to_check = {
    "Intent Labeled": STAGING_DIR / "intent_labeled.csv",
    "Train": STAGING_DIR / "train" / "intent_train.csv",
    "Validation": STAGING_DIR / "val" / "intent_val.csv",
    "Test": STAGING_DIR / "test" / "intent_test.csv",
}

print("========== ROW COUNT VERIFICATION ==========\n")

counts = {}

for name, path in files_to_check.items():
    df = pd.read_csv(path)
    counts[name] = len(df)

    print(f"{name:15}: {len(df):,} rows")

print("\n--------------------------------")

existing_total = (
    counts["Train"]
    + counts["Validation"]
    + counts["Test"]
)

print(f"Existing splits total : {existing_total:,}")
print(f"Intent labeled        : {counts['Intent Labeled']:,}")
print(f"Combined row count    : {existing_total + counts['Intent Labeled']:,}")

print("--------------------------------")

print("\nExpected existing total: 66,124")

if existing_total == 66124:
    print("✓ Existing train/val/test counts are correct.")
else:
    print("⚠ Existing train/val/test counts DO NOT match 66,124.")

========== ROW COUNT VERIFICATION ==========

Intent Labeled : 66,124 rows
Train          : 46,286 rows
Validation     : 9,919 rows
Test           : 9,919 rows

--------------------------------
Existing splits total : 66,124
Intent labeled        : 66,124
Combined row count    : 132,248
--------------------------------

Expected existing total: 66,124
✓ Existing train/val/test counts are correct.


## Kaggle metadata

In [20]:
from pathlib import Path

METADATA_PATH = STAGING_DIR / "dataset-metadata.json"

print("Metadata exists:", METADATA_PATH.exists())

if METADATA_PATH.exists():
    print("\n========== EXISTING METADATA ==========")
    print(METADATA_PATH.read_text())
else:
    print("\nNo dataset-metadata.json found in staging directory.")

Metadata exists: False

No dataset-metadata.json found in staging directory.


In [23]:
import json

METADATA_PATH = STAGING_DIR / "dataset-metadata.json"

metadata = {
    "title": "Processed Intent Data",
    "id": "prishabhkumar/processed-intent-data",
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ],
    "description": (
        "Processed email intent classification dataset containing "
        "train, validation, test splits and automatically labeled "
        "Enron email data."
    )
}

with open(METADATA_PATH, "w") as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

print("Created:")
print(METADATA_PATH)

print("\nMetadata:")
print(METADATA_PATH.read_text())

Created:
/kaggle/working/processed-intent-data/dataset-metadata.json

Metadata:
{
  "title": "Processed Intent Data",
  "id": "prishabhkumar/processed-intent-data",
  "licenses": [
    {
      "name": "CC0-1.0"
    }
  ],
  "description": "Processed email intent classification dataset containing train, validation, test splits and automatically labeled Enron email data."
}


In [24]:
print("========== FINAL STAGING STRUCTURE ==========")

for path in sorted(STAGING_DIR.rglob("*")):
    relative_path = path.relative_to(STAGING_DIR)

    if path.is_dir():
        print(f"DIR  {relative_path}")
    else:
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"FILE {relative_path} | {size_mb:.2f} MB")

========== FINAL STAGING STRUCTURE ==========
FILE dataset-metadata.json | 0.00 MB
FILE intent_labeled.csv | 143.44 MB
DIR  test
FILE test/intent_test.csv | 21.07 MB
DIR  train
FILE train/intent_train.csv | 101.21 MB
DIR  val
FILE val/intent_val.csv | 21.16 MB


## Final upload

In [25]:
!kaggle datasets version \
    -p /kaggle/working/processed-intent-data \
    -m "Added automatically labeled Enron intent dataset" \
    -t \
    -r zip

Starting upload for file test.zip
100%|██████████████████████████████████████| 7.73M/7.73M [00:01<00:00, 6.13MB/s]
Upload successful: test.zip (8MB)
Starting upload for file intent_labeled.csv
100%|████████████████████████████████████████| 143M/143M [00:04<00:00, 32.6MB/s]
Upload successful: intent_labeled.csv (143MB)
Starting upload for file train.zip
100%|██████████████████████████████████████| 37.1M/37.1M [00:01<00:00, 19.4MB/s]
Upload successful: train.zip (37MB)
Starting upload for file val.zip
100%|██████████████████████████████████████| 7.82M/7.82M [00:01<00:00, 6.41MB/s]
Upload successful: val.zip (8MB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/prishabhkumar/processed-intent-data


In [26]:
!kaggle datasets files prishabhkumar/processed-intent-data

name                         size  creationDate                
----------------------  ---------  --------------------------  
intent_labeled.csv      150407362  2026-08-18 12:15:24.520000  
test/intent_test.csv     22091464  2026-08-18 12:15:21.702000  
train/intent_train.csv  106129956  2026-08-18 12:15:24.430000  
val/intent_val.csv       22186102  2026-08-18 12:15:21.625000  
